## Extract Transform and Load (ETL) dos dados de movimentação de cargas do porto de Santos.
Disponibilizados pelo site da APS em: https://mensario.portodesantos.com.br/cargas/ \
O objetivo desse notebook é extrair a série temporal dos registros de movimentação de cargas contidos no arquivo .csv

Obs: Os dados estão em um formato "Long Table" precisaremos pivotar a agregar os dados para que fique em um período de registros mensais aos longo dos anos.

Referência: Python for Data Analysis, 3E (https://wesmckinney.com/book/)

In [1]:
#Importação da biblioteca necessária para esse procedimento:
import pandas as pd

In [2]:
df = pd.read_csv("../data/raw/exportacao_cargas.csv", encoding = 'utf-8')

In [3]:
display(df)

,TOTAL_TEU,NUMERO_VIAGEM,TIPO_NAVEGACAO,MOVIMENTO,ANO,TERMINAIS,TOTAL_TONELADAS,NATUREZA_CARGA,TOTAL_UNID,MES,CLASSENAVIO,SENTIDO,TIPO_INSTALACAO,BERCOS,MERCADORIAS,ANO_MES
0,16,515,LONGO CURSO,REMOÇÃO,2024,SANTOS BRASIL,"322,26",CARGA CONTEINERIZADA,16,2,PORTA-CONTAINERS,DESEMBARQUE,PORTO ORGANIZADO,SBR 2,CAFÉ,2024-02-01
1,2,860,LONGO CURSO,TRANSBORDO,2024,DPWORLD (EMBRAPORT),"24,26",CARGA CONTEINERIZADA,1,3,PORTA-CONTAINERS,DESEMBARQUE,TUP,DPW 1,PRODUTOS QUÍMICOS ORGÂNICOS,2024-03-01
2,20,1426,LONGO CURSO,TRANSBORDO,2024,BTP,"245,91",CARGA CONTEINERIZADA,10,4,PORTA-CONTAINERS,DESEMBARQUE,PORTO ORGANIZADO,BTP 03,OUTRAS MERCADORIAS,2024-04-01
3,4,1372,LONGO CURSO,TRANSBORDO,2024,DPWORLD (EMBRAPORT),"36,537",CARGA CONTEINERIZADA,2,4,PORTA-CONTAINERS,EMBARQUE,TUP,DPW 1,MOTOCICLETAS,2024-04-01
4,31,1234,LONGO CURSO,CONVENCIONAL,2024,BTP,"658,99",CARGA CONTEINERIZADA,31,3,PORTA-CONTAINERS,EMBARQUE,PORTO ORGANIZADO,BTP 02,CAFÉ,2024-03-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1081271,0,1,LONGO CURSO,CONVENCIONAL,2005,OUTROS,79919,GRANEL LIQUIDO,0,1,OUTROS,EMBARQUE,OUTROS,ABASTECIMENTO,BUNKER (O.COMBUSTIVEL),2005-01-01
1081272,0,1,LONGO CURSO,CONVENCIONAL,2005,OUTROS,84572,GRANEL LIQUIDO,0,4,OUTROS,EMBARQUE,OUTROS,ABASTECIMENTO,BUNKER (O.COMBUSTIVEL),2005-04-01
1081273,0,1,LONGO CURSO,CONVENCIONAL,2006,OUTROS,85221,GRANEL LIQUIDO,0,2,OUTROS,EMBARQUE,OUTROS,ABASTECIMENTO,BUNKER (O.COMBUSTIVEL),2006-02-01
1081274,0,1,LONGO CURSO,CONVENCIONAL,2006,OUTROS,89970,GRANEL LIQUIDO,0,4,OUTROS,EMBARQUE,OUTROS,ABASTECIMENTO,BUNKER (O.COMBUSTIVEL),2006-04-01


In [4]:
#Filtrando somente as colunas necessárias para a formação da série temporal
filtrar_colunas = ['ANO', 'MES', 'TOTAL_TONELADAS', 'MERCADORIAS']
df_filtrado = df[filtrar_colunas].copy()

In [5]:
display(df_filtrado) #aqui teremos os registros de data, tipo de mercadoria e toneladas movimentadas

,ANO,MES,TOTAL_TONELADAS,MERCADORIAS
0,2024,2,"322,26",CAFÉ
1,2024,3,"24,26",PRODUTOS QUÍMICOS ORGÂNICOS
2,2024,4,"245,91",OUTRAS MERCADORIAS
3,2024,4,"36,537",MOTOCICLETAS
4,2024,3,"658,99",CAFÉ
...,...,...,...,...
1081271,2005,1,79919,BUNKER (O.COMBUSTIVEL)
1081272,2005,4,84572,BUNKER (O.COMBUSTIVEL)
1081273,2006,2,85221,BUNKER (O.COMBUSTIVEL)
1081274,2006,4,89970,BUNKER (O.COMBUSTIVEL)


In [6]:
#Passando os valores de toneladas para tipo float e trocando separador de milhares ',' por '.'
df_filtrado['TOTAL_TONELADAS'] = pd.to_numeric(
    df_filtrado['TOTAL_TONELADAS'].str.replace(',', '.', regex=False),
    errors='coerce'
    )
print(df_filtrado['TOTAL_TONELADAS'].dtype)

#Verificando se ficaram valores Not a Number (NaN) 
nans = df_filtrado['TOTAL_TONELADAS'].isna().sum()
print(f"NaNs introduzidos na conversão: {nans}")

float64
NaNs introduzidos na conversão: 0


In [7]:
display(df_filtrado)

,ANO,MES,TOTAL_TONELADAS,MERCADORIAS
0,2024,2,322.260,CAFÉ
1,2024,3,24.260,PRODUTOS QUÍMICOS ORGÂNICOS
2,2024,4,245.910,OUTRAS MERCADORIAS
3,2024,4,36.537,MOTOCICLETAS
4,2024,3,658.990,CAFÉ
...,...,...,...,...
1081271,2005,1,79919.000,BUNKER (O.COMBUSTIVEL)
1081272,2005,4,84572.000,BUNKER (O.COMBUSTIVEL)
1081273,2006,2,85221.000,BUNKER (O.COMBUSTIVEL)
1081274,2006,4,89970.000,BUNKER (O.COMBUSTIVEL)


In [8]:
#Introduzindo nova coluna TOTAL_MENSAL que irá quantificar cada produto por mês e ano
#qualquer linha que tiver o mesmo par (mes/ano) também terá os mesmos valores em TOTAL_MENSAL
df_filtrado['TOTAL_MENSAL'] = df_filtrado.groupby(['ANO', 'MES'])['TOTAL_TONELADAS'].transform('sum')

# Verificação
display(df_filtrado)
display(df_filtrado.dtypes)

,ANO,MES,TOTAL_TONELADAS,MERCADORIAS,TOTAL_MENSAL
0,2024,2,322.260,CAFÉ,1.431857e+07
1,2024,3,24.260,PRODUTOS QUÍMICOS ORGÂNICOS,1.611037e+07
2,2024,4,245.910,OUTRAS MERCADORIAS,1.469959e+07
3,2024,4,36.537,MOTOCICLETAS,1.469959e+07
4,2024,3,658.990,CAFÉ,1.611037e+07
...,...,...,...,...,...
1081271,2005,1,79919.000,BUNKER (O.COMBUSTIVEL),5.027936e+06
1081272,2005,4,84572.000,BUNKER (O.COMBUSTIVEL),5.850636e+06
1081273,2006,2,85221.000,BUNKER (O.COMBUSTIVEL),5.321604e+06
1081274,2006,4,89970.000,BUNKER (O.COMBUSTIVEL),6.450283e+06


ANO                  int64
MES                  int64
TOTAL_TONELADAS    float64
MERCADORIAS         object
TOTAL_MENSAL       float64
dtype: object

In [9]:
#Verificação dos tipos de cargas na coluna MERCADORIAS
lista_cargas = df_filtrado['MERCADORIAS'].unique()
print(lista_cargas)

['CAFÉ' 'PRODUTOS QUÍMICOS ORGÂNICOS' 'OUTRAS MERCADORIAS' 'MOTOCICLETAS'
 'NAFTA' 'ÓLEO DE ORIGEM VEGETAL' 'SEM CARGAS'
 'PRODUTOS DIVERSOS DA INDÚSTRIA QUÍMICA' 'ALGODÃO' 'SOLVENTES'
 'SUCOS CÍTRICOS' 'FERTILIZANTES (MISTURAS)' 'FERTILIZANTES POTÁSSICOS'
 'FERTILIZANTES NITROGENADOS' 'VEÍCULOS AUTOMÓVEIS (USOS ESPECIAIS)'
 'PRODUTOS QUÍMICOS INORGÂNICOS' 'FERTILIZANTES'
 'PEIXES E OUTROS AQUÁTICOS' 'AÇÚCAR' 'CARNE DE AVES' 'CARNES DIVERSAS'
 'CARNE BOVINA' 'PRODUTOS SIDERÚRGICOS' 'SAL' 'ENXOFRE' 'OUTROS VEÍCULOS'
 'CELULOSE' 'METANOL'
 'VEÍCULOS AUTOMÓVEIS (TRANSP. DE PASS. (<10 PASS.)+ VEIC. CORRIDA)'
 'CARNE SUÍNA' 'COMBUSTÍVEIS ÓLEOS E PRODUTOS MINERAIS' 'FARELO DE SOJA'
 'VEÍCULOS AUTOMÓVEIS (TRANSP. DE MERCADORIAS)' 'ETANOL'
 'PETRÓLEO E DERIVADOS' 'CARNE OVINA/CAPRINA' 'MILHO' 'SODA CÁUSTICA'
 'CARVÃO'
 'VEÍCULOS AUTOMÓVEIS (ESPECIAIS, P TRANSP. DE CARGAS EM FÁBRICAS, ARMAZÉNS...)'
 'VEÍCULOS AUTOMÓVEIS (CHASSIS COM MOTOR)' 'OUTROS SUCOS'
 'VEÍCULOS AUTOMÓVEIS (TRATORES)' 'ANIM

In [10]:
#Verificação dos tipos de carga com maior quantidade de movimentação, os 10 maiores:
ranking = (
    df_filtrado
    .groupby('MERCADORIAS')['TOTAL_TONELADAS']
    .sum()
    .sort_values(ascending=False)
)
print(ranking.head(10))

MERCADORIAS
OUTRAS MERCADORIAS             5.972902e+08
AÇÚCAR                         3.726214e+08
SOJA EM GRÃOS                  3.477256e+08
MILHO                          2.108713e+08
FARELO DE SOJA                 1.073321e+08
CELULOSE                       8.476397e+07
PRODUTOS QUÍMICOS ORGÂNICOS    6.778048e+07
ÓLEO DIESEL                    6.300103e+07
ÓLEO COMBUSTÍVEL               5.122767e+07
SUCOS CÍTRICOS                 4.454443e+07
Name: TOTAL_TONELADAS, dtype: float64


In [11]:
#Selecionaremos os 5 tipos com maior quantidade de movimentação
#Esses mesmos tipos de cargas são interessante de explorar em um série temporal devido a influência das safras nesses commodities
tipos = ['SOJA EM GRÃOS', 'OUTRAS MERCADORIAS', 'AÇÚCAR', 'FARELO DE SOJA', 'MILHO']
df_mercadorias = df_filtrado[df_filtrado['MERCADORIAS'].isin(tipos)].copy()
display(df_mercadorias) #observe a redução no número de linhas abaixo da visualização do data frame:

,ANO,MES,TOTAL_TONELADAS,MERCADORIAS,TOTAL_MENSAL
2,2024,4,245.910,OUTRAS MERCADORIAS,1.469959e+07
7,2024,2,7928.025,OUTRAS MERCADORIAS,1.431857e+07
9,2024,3,471.692,OUTRAS MERCADORIAS,1.611037e+07
13,2024,5,8565.180,OUTRAS MERCADORIAS,1.588925e+07
17,2024,5,32.125,OUTRAS MERCADORIAS,1.588925e+07
...,...,...,...,...,...
1080243,2010,5,52.700,OUTRAS MERCADORIAS,8.988241e+06
1080244,2010,7,547.171,OUTRAS MERCADORIAS,8.603807e+06
1080251,2010,7,27.100,OUTRAS MERCADORIAS,8.603807e+06
1080254,2010,8,16.190,OUTRAS MERCADORIAS,9.547551e+06


In [12]:
#Rotacionaremos o dataframe para obter um formato em linhas de cada mercadoria somada mes unicamente a cada ano
df_st = df_mercadorias.pivot_table(
    index=['ANO','MES'],
    columns='MERCADORIAS',
    values='TOTAL_TONELADAS',
    aggfunc='sum',
    fill_value=0 #aqui estamos tratando a ausência de um registro em qualquer período como 0 e não como dado faltante, então é um ponto de atenção necessário para qualquer replicação desse código,
)
df_st = df_st.sort_index(level=['ANO','MES']).reset_index()
df_st.columns.name = None

In [13]:
display(df_st)

,ANO,MES,AÇÚCAR,FARELO DE SOJA,MILHO,OUTRAS MERCADORIAS,SOJA EM GRÃOS
0,2005,1,581383.455,148641.973,67968.749,1675915.136,107966.285
1,2005,2,878015.791,207084.350,11623.584,1709254.262,390955.911
2,2005,3,610138.354,186525.305,12014.295,1730293.750,896454.461
3,2005,4,764282.986,340274.229,7687.881,1860562.606,858345.463
4,2005,5,1007703.226,330537.573,12814.725,2027806.818,1062494.325
...,...,...,...,...,...,...,...
250,2025,11,2608045.205,972150.444,2408073.789,3073654.839,1170675.035
251,2025,12,1648997.531,719107.585,2594208.417,2975549.057,1012862.467
252,2026,1,1561010.948,867668.554,1151919.295,2886656.038,699950.201
253,2026,2,1477114.383,625398.725,74968.470,2661940.849,3235398.809


In [14]:
#Juntaremos ao data frame de série temporal a variável porto com a movimentação total de todas as mercadorias 
porto = (
    df_filtrado
    .groupby(['ANO','MES'])['TOTAL_TONELADAS']
    .sum()
    .reset_index()
    .rename(columns={'TOTAL_TONELADAS': 'porto'})
)
df_st = df_st.merge(porto, on=['ANO','MES'], how='left')

In [15]:
df_st.rename(columns={'ANO':'ano','MES':'mes','AÇÚCAR':'acucar','FARELO DE SOJA':'farelo_soja','MILHO':'milho','OUTRAS MERCADORIAS':'outros','SOJA EM GRÃOS':'graos_soja'}, inplace=True)

In [16]:
display(df_st)

,ano,mes,acucar,farelo_soja,milho,outros,graos_soja,porto
0,2005,1,581383.455,148641.973,67968.749,1675915.136,107966.285,5.027936e+06
1,2005,2,878015.791,207084.350,11623.584,1709254.262,390955.911,5.316605e+06
2,2005,3,610138.354,186525.305,12014.295,1730293.750,896454.461,5.899909e+06
3,2005,4,764282.986,340274.229,7687.881,1860562.606,858345.463,5.850636e+06
4,2005,5,1007703.226,330537.573,12814.725,2027806.818,1062494.325,6.826361e+06
...,...,...,...,...,...,...,...,...
250,2025,11,2608045.205,972150.444,2408073.789,3073654.839,1170675.035,1.612665e+07
251,2025,12,1648997.531,719107.585,2594208.417,2975549.057,1012862.467,1.473902e+07
252,2026,1,1561010.948,867668.554,1151919.295,2886656.038,699950.201,1.272810e+07
253,2026,2,1477114.383,625398.725,74968.470,2661940.849,3235398.809,1.317256e+07


In [17]:
#Criaremos um index como tipo datetime
df_st['data'] = pd.to_datetime(dict(year=df_st['ano'], month=df_st['mes'], day=1))
df_st = df_st.set_index('data')
df_st.index = pd.DatetimeIndex(df_st.index, freq='MS')
df_st = df_st.drop(columns=['ano', 'mes'])
display(df_st.tail())

,acucar,farelo_soja,milho,outros,graos_soja,porto
data,,,,,,
2025-11-01,2608045.205,972150.444,2408073.789,3073654.839,1170675.035,1.612665e+07
2025-12-01,1648997.531,719107.585,2594208.417,2975549.057,1012862.467,1.473902e+07
2026-01-01,1561010.948,867668.554,1151919.295,2886656.038,699950.201,1.272810e+07
2026-02-01,1477114.383,625398.725,74968.470,2661940.849,3235398.809,1.317256e+07
2026-03-01,1167837.505,998819.742,4121.160,3168250.752,6086504.453,1.688558e+07


In [18]:
df_st.to_csv('../data/processed/serietemporal.csv')